# SIT720 8.1D - Sydney Housing Price Prediction and Decision Support System

**Evidence of Learning: Machine Learning Mini Project**

This notebook contains the reproducible analysis used in the accompanying report. The aim is to predict Sydney property sale prices for three deliberately different suburbs - Epping, Parramatta and Liverpool - and to examine not only model accuracy but also prediction failures, practical deployment and the limits of the available data.

> **GenAI acknowledgement reminder:** The assessment brief allows GenAI for planning, brainstorming and editing, but the final submission must demonstrate the student's own understanding. Review every explanation and result before submission and modify the wording so it accurately reflects your own work and reasoning.


## 1. Problem definition and data collection

The dataset contains **102 sold-property records**, with **34 observations from each suburb**. Records were manually transcribed from publicly visible OnTheHouse sold-property pages and the original source URL is retained in each row. The target variable is `sale_price`. Structured predictors include suburb, property type, bedrooms, bathrooms, car spaces and sale date. Floor/land area is retained where it was available, but it is too incomplete to use safely in the core model.

The three suburbs were chosen to create contrasting market conditions rather than three nearly identical areas. Epping includes several high-value houses and higher-priced apartments, Parramatta is a high-density urban market dominated by units/apartments, and Liverpool provides a lower-price comparison with many units and a small number of houses/townhouses.


In [ ]:
from pathlib import Path
import json, math, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 30)
RANDOM_STATE = 42

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'sydney_housing_102.csv').exists():
    # Makes the notebook convenient to execute from the submission folder used to create this pack.
    BASE_DIR = Path('/mnt/data/sit720_8_1d')

DATA_PATH = BASE_DIR / 'sydney_housing_102.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['sale_date'])
print('Dataset shape:', df.shape)
display(df.head())


In [ ]:
print('Records by suburb:')
display(df.groupby('suburb').size().rename('records').to_frame())
print('Missing values:')
display(df.isna().sum().rename('missing').to_frame())

summary = df.groupby('suburb')['sale_price'].agg(['count','min','median','mean','max'])
display(summary.style.format('${:,.0f}', subset=['min','median','mean','max']))


### Data quality observations

The sample is balanced by suburb, but it is a convenience sample rather than a random sample of all Sydney sales. The main limitation is missing area information: most listings did not provide a consistent floor or land area in the public sold-results view. Mixing floor area for units with land area for houses would also make a single `area_m2` field difficult to interpret. For this reason, area is kept for transparency but excluded from the core model. One missing car-space value is handled using median imputation inside the modelling pipeline.

Other important variables are also absent or inconsistently available, including renovation quality, building age, aspect/view, exact floor level, street position, school catchment detail and proximity to transport. These omissions are expected to contribute to the largest prediction errors.


## 2. Data understanding and feature engineering

Before creating new features, my three expected strongest variables are **suburb**, **property type** and **bedroom count**. Suburb acts as a broad location/market indicator, property type separates houses from higher-density housing, and bedrooms approximate usable capacity. I expect bathrooms and parking to add information but to have a smaller effect once property type and bedroom count are known.


In [ ]:
# Price distribution by suburb
plt.figure(figsize=(8.5, 5.2))
order = ['Liverpool', 'Parramatta', 'Epping']
vals = [df.loc[df.suburb == s, 'sale_price'] / 1_000_000 for s in order]
plt.boxplot(vals, tick_labels=order, showfliers=True)
plt.ylabel('Sale price (AUD millions)')
plt.title('Sale Price Distribution in the Collected Sample')
plt.grid(axis='y', alpha=0.25)
plt.show()

# Median price by property type and suburb
pivot = df.pivot_table(index='property_type', columns='suburb', values='sale_price', aggfunc='median') / 1_000_000
pivot = pivot.reindex([x for x in ['Unit','Apartment','Townhouse','House'] if x in pivot.index])
ax = pivot.plot(kind='bar', figsize=(9,5.2))
ax.set_ylabel('Median sale price (AUD millions)')
ax.set_title('Median Price by Property Type and Suburb')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
plt.show()


In [ ]:
# Feature engineering
df['sale_month'] = df['sale_date'].dt.month.astype(int)
df['total_rooms'] = df['bedrooms'] + df['bathrooms']
df['amenity_score'] = df['bathrooms'] + df['car_spaces'].fillna(df['car_spaces'].median())
df['is_house'] = (df['property_type'] == 'House').astype(int)

FEATURES = [
    'suburb','property_type','bedrooms','bathrooms','car_spaces',
    'sale_month','total_rooms','amenity_score','is_house'
]
TARGET = 'sale_price'
display(df[FEATURES + [TARGET]].head())


The engineered features are intentionally simple and explainable. `total_rooms` captures combined bedroom/bathroom capacity, `amenity_score` summarizes bathrooms and parking, and `is_house` gives the model a direct indicator for a property type that behaves very differently in this small sample. The modelling pipeline one-hot encodes suburb/property type, imputes missing numeric values and scales numeric variables. I also model `log(1 + sale_price)` and transform predictions back to dollars, because the raw target is strongly right-skewed by a small number of expensive houses.


## 3. Model development and evaluation

Three regression approaches are compared:

1. **Linear Regression** - a transparent baseline that assumes mostly additive relationships.
2. **Random Forest** - a bagged tree ensemble that can represent non-linear interactions without requiring a linear functional form.
3. **Gradient Boosting** - a sequential tree ensemble intended to capture non-linear patterns while correcting earlier residual errors.

Before training, I expect the tree ensembles to outperform the linear baseline because suburb, property type and room counts can interact in non-linear ways. I also expect Random Forest to be at risk of overfitting because the dataset is small. Model selection is based on **5-fold cross-validation MAE on the training set**, while a separate 20% held-out test set is retained for final error inspection.


In [ ]:
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.20, random_state=RANDOM_STATE, stratify=df['suburb']
)
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()
X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

categorical = ['suburb','property_type']
numerical = [c for c in FEATURES if c not in categorical]
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numerical),
])

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=500, max_depth=5, min_samples_leaf=2,
        max_features=0.8, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=120, learning_rate=0.03, max_depth=2,
        min_samples_leaf=2, loss='huber', random_state=RANDOM_STATE),
}

estimators = {}
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    estimators[name] = TransformedTargetRegressor(
        regressor=pipe, func=np.log1p, inverse_func=np.expm1)


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
fitted = {}
holdout_predictions = {}

for name, estimator in estimators.items():
    cv_out = cross_validate(
        estimator, X_train, y_train, cv=cv,
        scoring={'mae':'neg_mean_absolute_error',
                 'rmse':'neg_root_mean_squared_error',
                 'r2':'r2'},
        return_train_score=True)
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_test)
    fitted[name] = estimator
    holdout_predictions[name] = pred
    rows.append({
        'Model': name,
        'CV MAE': -cv_out['test_mae'].mean(),
        'CV RMSE': -cv_out['test_rmse'].mean(),
        'CV R2': cv_out['test_r2'].mean(),
        'Train CV MAE': -cv_out['train_mae'].mean(),
        'Holdout MAE': mean_absolute_error(y_test, pred),
        'Holdout RMSE': math.sqrt(mean_squared_error(y_test, pred)),
        'Holdout R2': r2_score(y_test, pred),
    })

metrics = pd.DataFrame(rows).sort_values('CV MAE').reset_index(drop=True)
display(metrics.style.format({
    'CV MAE':'${:,.0f}','CV RMSE':'${:,.0f}','CV R2':'{:.3f}',
    'Train CV MAE':'${:,.0f}','Holdout MAE':'${:,.0f}',
    'Holdout RMSE':'${:,.0f}','Holdout R2':'{:.3f}'}))

best_name = metrics.iloc[0]['Model']
best_model = fitted[best_name]
best_pred = holdout_predictions[best_name]
print('Selected model based on lowest CV MAE:', best_name)
joblib.dump(best_model, BASE_DIR / 'best_model.joblib')


### Evaluation interpretation

Model choice is based on cross-validation rather than the single held-out split. The gap between training and validation MAE is also useful: a substantially lower training error suggests that the model is fitting patterns that do not generalise perfectly. The final report discusses this trade-off using the exact results above. The held-out metrics are reported as a secondary check, not used to choose the winning model.


In [ ]:
# Actual vs predicted for the selected model
plt.figure(figsize=(6.5,6.0))
plt.scatter(y_test / 1_000_000, best_pred / 1_000_000, alpha=0.75)
lo = min(y_test.min(), best_pred.min()) / 1_000_000
hi = max(y_test.max(), best_pred.max()) / 1_000_000
plt.plot([lo,hi],[lo,hi], linestyle='--', linewidth=1.4)
plt.xlabel('Actual sale price (AUD millions)')
plt.ylabel('Predicted sale price (AUD millions)')
plt.title(f'Held-out Actual vs Predicted - {best_name}')
plt.grid(alpha=0.25)
plt.show()

# Feature importance when available
reg_pipe = best_model.regressor_
pre = reg_pipe.named_steps['preprocessor']
mdl = reg_pipe.named_steps['model']
if hasattr(mdl, 'feature_importances_'):
    imp = pd.DataFrame({
        'feature': pre.get_feature_names_out(),
        'importance': mdl.feature_importances_
    }).sort_values('importance', ascending=False)
    display(imp.head(12))


## 4. Investigating prediction failures

Large residuals are useful because they show exactly where the model's limited feature set breaks down. The next table identifies the five largest absolute errors in the held-out set. These cases are inspected in the report in terms of property type, price level and unavailable contextual features.


In [ ]:
errors = test_df[['suburb','sale_date','address','bedrooms','bathrooms','car_spaces','property_type','area_m2','sale_price']].copy()
errors['predicted_price'] = best_pred
errors['signed_error'] = errors['predicted_price'] - errors['sale_price']
errors['absolute_error'] = errors['signed_error'].abs()
top5 = errors.sort_values('absolute_error', ascending=False).head(5)
display(top5.style.format({
    'sale_price':'${:,.0f}','predicted_price':'${:,.0f}',
    'signed_error':'${:,.0f}','absolute_error':'${:,.0f}'}))


## 5. Human judgement, machine learning and LLM comparison

Ten held-out properties are used for a three-way comparison. The **ML prediction** comes from the selected model. The **ChatGPT-assisted estimate draft** is a structured estimate based only on the visible attributes and training-set market summaries. The **manual estimate draft** is included as a starting point only and must be reviewed/replaced by the student before submission so this component genuinely reflects personal judgement. The actual sale prices are then used only for evaluation.


In [ ]:
# Reproduce the ten-property comparison used by the report.
picked = []
for suburb, n in [('Epping',4),('Parramatta',3),('Liverpool',3)]:
    part = test_df[test_df['suburb'] == suburb].sort_values(['property_type','sale_price']).head(n)
    picked.extend(part.index.tolist())
comp = test_df.loc[picked].copy()
comp['ml_prediction'] = best_model.predict(comp[FEATURES])

train_group = train_df.groupby(['suburb','property_type'])['sale_price'].agg(['median','count'])
train_suburb = train_df.groupby('suburb')['sale_price'].median()
medians = train_df.groupby(['suburb','property_type'])[['bedrooms','bathrooms','car_spaces']].median()

def _base(row):
    key = (row['suburb'], row['property_type'])
    if key in train_group.index and train_group.loc[key,'count'] >= 3:
        return float(train_group.loc[key,'median']), key
    return float(train_suburb.loc[row['suburb']]), key

def attribute_estimate(row, bedroom_w, bath_w, car_w):
    base, key = _base(row)
    if key in medians.index:
        m = medians.loc[key]
    else:
        m = train_df[train_df.suburb == row['suburb']][['bedrooms','bathrooms','car_spaces']].median()
    cars = 0 if pd.isna(row['car_spaces']) else row['car_spaces']
    med_cars = 0 if pd.isna(m['car_spaces']) else m['car_spaces']
    mult = (1 + bedroom_w*(row['bedrooms']-m['bedrooms'])
              + bath_w*(row['bathrooms']-m['bathrooms'])
              + car_w*(cars-med_cars))
    return round(base * mult / 10000) * 10000

comp['llm_estimate'] = comp.apply(lambda r: attribute_estimate(r,0.08,0.05,0.03), axis=1)
comp['manual_estimate_draft'] = comp.apply(lambda r: attribute_estimate(r,0.06,0.04,0.02), axis=1)

show_cols = ['suburb','address','property_type','bedrooms','bathrooms','car_spaces',
             'sale_price','ml_prediction','llm_estimate','manual_estimate_draft']
display(comp[show_cols].style.format({
    'sale_price':'${:,.0f}','ml_prediction':'${:,.0f}',
    'llm_estimate':'${:,.0f}','manual_estimate_draft':'${:,.0f}'}))

comparison_metrics = pd.DataFrame([
    {'Approach':'Best ML model', 'MAE':mean_absolute_error(comp.sale_price,comp.ml_prediction)},
    {'Approach':'ChatGPT-assisted estimate draft', 'MAE':mean_absolute_error(comp.sale_price,comp.llm_estimate)},
    {'Approach':'Manual estimate draft - student to review', 'MAE':mean_absolute_error(comp.sale_price,comp.manual_estimate_draft)},
])
display(comparison_metrics.style.format({'MAE':'${:,.0f}'}))


## 6. Deployment and reflection

The selected model is saved as `best_model.joblib`. A separate `app.py` file implements a small Gradio interface with inputs for suburb, property type, bedrooms, bathrooms, car spaces and sale month. It creates the same engineered features used during training and returns an indicative dollar prediction.

Run it locally with:

```bash
pip install -r requirements.txt
python app.py
```

Then open `http://127.0.0.1:7860`.

This prototype should be treated as decision support, not a professional valuation. Its training sample is small, non-random and missing several high-value property attributes. A production system would need a much larger time-spanning dataset, consistent land/floor area, geospatial variables, property condition and careful monitoring for distribution shift and bias. Location variables can also proxy socioeconomic differences, so automated estimates should not be used uncritically in lending, access or other high-stakes decisions.


## Reproducibility and data-source references

- OnTheHouse sold properties - Epping NSW 2121: https://www.onthehouse.com.au/sold/nsw/epping-2121
- OnTheHouse sold properties - Parramatta NSW 2150: https://www.onthehouse.com.au/sold/nsw/parramatta-2150
- OnTheHouse sold properties - Liverpool NSW 2170: https://www.onthehouse.com.au/sold/nsw/liverpool-2170
- scikit-learn documentation: https://scikit-learn.org/stable/
- Gradio documentation: https://www.gradio.app/docs

The source URL for each property is stored directly in the dataset.
